# Solcore Wavelength Sweep Data Processing

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import json
import re

In [ ]:
# Constants
Sim1Path = "Simulations/Simulation1_WavelengthSweep"
Sim2Path = "Simulations/Simulation2_WavelengthSweepWithPerturbance"
Sim3Path = "Simulations/Simulation3_WavelengthSweepWithWaveWithPerturbance"

## Debugging
Debug = False

## Folder Creation
if (not os.path.exists("Processed")):
    os.mkdir("Processed")

if (not os.path.exists("Processed/Simulation1")):
    os.mkdir("Processed/Simulation1")
    
if (not os.path.exists("Processed/Simulation2")):
    os.mkdir("Processed/Simulation2")

if (not os.path.exists("Processed/Simulation3")):
    os.mkdir("Processed/Simulation3")

## Merge the Angle JSON Files for Sim2 and Sim3

In [ ]:
# List out the JSON Files
Sim2Files = [file for file in os.listdir(Sim2Path) if os.path.isfile(os.path.join(Sim2Path, file)) and file != "FilePaths.json"]
Sim3Files = [file for file in os.listdir(Sim3Path) if os.path.isfile(os.path.join(Sim3Path, file)) and file != "FilePaths.json"]

print("Simulation 2 Files")
for file in Sim2Files:
    print(file)

print("-" * 30)

print("Simulation 3 Files")
for file in Sim3Files:
    print(file)


In [ ]:
# Merge the Files
def CombineJSON (files: list[str], path: str) -> json:
    
    combinedJson: json = {}
    
    for file in files:
        cleanedAngle = file.removeprefix("FilePaths_").removesuffix(".json")

        with open(os.path.join(path, file), "r") as jsonFile:
            data = json.load(jsonFile)
            
        combinedJson[cleanedAngle] = data
        
    return combinedJson
    
# Create new JSON Objects
Sim2JSON: json = CombineJSON(Sim2Files, Sim2Path)
Sim3JSON: json = CombineJSON(Sim3Files, Sim3Path)

with open(os.path.join(Sim2Path, "FilePaths.json"), "w") as file:
    json.dump(Sim2JSON, file, indent=4)

with open(os.path.join(Sim3Path, "FilePaths.json"), "w") as file:
    json.dump(Sim3JSON, file, indent=4)

# Utility Functions

In [ ]:
# Utility functions
def GetTRALambda(filepath: str):
    
    pattern = r"Wavelength_([\d.])+"
    
    fileJSON : json = {}
    
    with open(filepath, "r") as file:
        fileJSON = json.load(file)
        
    fileJSON = fileJSON["Stats"]
        
    name = fileJSON["Name"]
    startPower = float(fileJSON["StartPower"])
    capturedPower = float(fileJSON["CapturedPower"])
    destroyedPower = float(fileJSON["DestroyedPower"])
    lostPower = float(fileJSON["LostPower"])
    
    match = re.search(pattern, name)
    
    if not match:
        raise ValueError(f"Could not find a Matching Wavelength in FileName : {name}")
    
    wavelength = float(match.group(0).removeprefix("Wavelength_"))
    
    transmittance = capturedPower / startPower
    reflectance = lostPower / startPower
    absorbance = destroyedPower / startPower
    
    return (wavelength, transmittance, reflectance, absorbance)

def CreatePerturbanceDataFrame(perturbanceJSON : json) -> pd.DataFrame:
    dataframe = pd.DataFrame(columns=["Wavelength", "Transmittance", "Reflectance", "Absorbance"])
    
    for key in perturbanceJSON.keys():
        
        total = np.zeros(4)
        n = len(perturbanceJSON[key])
        
        for file in perturbanceJSON[key]:
            
            total += np.array(GetTRALambda(file + ".json"))
        
        dataframe.loc[len(dataframe)] = total / n
    
    return dataframe

# Process Simulation 1

In [ ]:
# Load Simulation 1 JSON
Sim1FilePathsPath = os.path.join(Sim1Path, "FilePaths.json")

Sim1FilePathsJSON: json = {}

with open(Sim1FilePathsPath, "r") as jsonFile:
    Sim1FilePathsJSON = json.load(jsonFile)

print(json.dumps(Sim1FilePathsJSON, indent=4))

In [ ]:
def CreateAngleDataFrame(angleJSON : json) -> pd.DataFrame:
    dataframe = pd.DataFrame(columns=["Wavelength", "Transmittance", "Reflectance", "Absorbance"])
    
    for file in angleJSON:
        dataframe.loc[len(dataframe)] = GetTRALambda(file + ".json")
    
    return dataframe
    
for key in Sim1FilePathsJSON.keys():
    simDataFrame = CreateAngleDataFrame(Sim1FilePathsJSON[key])
    
    angle = key.removeprefix("Angle_")
    
    simDataFrame.to_csv(f"Processed/Simulation1/MothEye_Raytracing_{angle}_Regular.csv")
    
    if (Debug):
        print(key)
        display(simDataFrame)


# Process Simulation 2

In [ ]:
# Load Simulation 2 JSON
Sim2FilePathsPath = os.path.join(Sim2Path, "FilePaths.json")

Sim2FilePathsJSON: json = {}

with open(Sim2FilePathsPath, "r") as jsonFile:
    Sim2FilePathsJSON = json.load(jsonFile)

print(json.dumps(Sim2FilePathsJSON, indent=4))

In [ ]:
# Process Simulation 2
for key in Sim2FilePathsJSON.keys():
    for pertKey in Sim2FilePathsJSON[key].keys():
        simDataFrame = CreatePerturbanceDataFrame(Sim2FilePathsJSON[key][pertKey])
        
        angle = key.removeprefix("Angle_")
        perturbance = pertKey.removeprefix("PerturbanceDev_")
        
        simDataFrame.to_csv(f"Processed/Simulation2/MothEye_Raytracing_{angle}_Perturbance_{perturbance}_Sim2.csv")
        
        if (Debug):
            print(key)
            display(simDataFrame)


# Process Simulation 3

In [ ]:
# Load Simulation 3 JSON
Sim3FilePathsPath = os.path.join(Sim3Path, "FilePaths.json")

Sim3FilePathsJSON: json = {}

with open(Sim3FilePathsPath, "r") as jsonFile:
    Sim3FilePathsJSON = json.load(jsonFile)

print(json.dumps(Sim3FilePathsJSON, indent=4))

In [ ]:
# Process Simulation 3
for key in Sim3FilePathsJSON.keys():
    for pertKey in Sim3FilePathsJSON[key].keys():
        simDataFrame = CreatePerturbanceDataFrame(Sim3FilePathsJSON[key][pertKey])
        
        angle = key.removeprefix("Angle_")
        perturbance = pertKey.removeprefix("PerturbanceDev_")
        
        simDataFrame.to_csv(f"Processed/Simulation3/MothEye_Raytracing_{angle}_Perturbance_{perturbance}_Sim3.csv")
        
        if (Debug):
            print(key)
            display(simDataFrame)
